# RepairScope-X — Notebook 12
## Dream-7B Self-Repair Surface: Repairing Dream's Own Failures

Notebook 11 produced the missing model-native failure corpus:

- **96/96** Dream generations completed;
- **52 task failures** were observed;
- failures covered all **3 datasets**;
- **28 unique problem IDs** failed at least once;
- **24 problem IDs** failed under both independent Dream generation seeds;
- `DREAM_OWN_FAILURE_CORPUS_READY = true`.

This notebook now asks the central question directly:

> **Can a diffusion language model selectively reopen and repair its own failed output, and how does repairability change with intervention scope?**

### Balanced self-repair cohort

A deterministic pre-repair selection rule chooses **12 Dream-origin failures**:

- 4 Countdown
- 4 GSM8K
- 4 SVAMP

Within each dataset, persistent failures are prioritized and then ranked by SHA-256 using selection seed `3400`. No repair outcome is used for selection. One exact failed Dream response is retained per problem ID.

Because only two SVAMP problem IDs failed under both source-generation seeds, the SVAMP cohort contains those two persistent failures plus two one-seed Dream failures. Countdown and GSM8K selections are persistent failures.

### Self-repair actions

The failed Dream response is re-tokenized and used as the repair canvas.

- `open_last_25`: preserve first 75%, remask last 25%.
- `open_last_50`: preserve first 50%, remask last 50%.
- `open_last_75`: preserve first 25%, remask last 75%.
- `global`: remask the entire response; matched-NFE fresh regeneration control.

All conditions use exactly **64 reverse denoising steps**.

### Experimental matrix

12 Dream-origin failures × 4 actions × 5 new repair seeds = **240 DLM self-repair trials**

Repair seeds:

`3401, 3402, 3403, 3404, 3405`

The notebook reports:

- strict / numeric / task-aware repair success;
- actual NFE, wall time and peak VRAM;
- preservation;
- per-failure repair surfaces;
- best fixed action vs per-instance oracle;
- DLM Oracle Repair Headroom;
- same-wrong numeric recurrence;
- cluster-bootstrap 95% confidence intervals;
- paired selective-vs-global tests with Holm correction.

This remains an exploratory **NF4 4-bit** condition for the available T4-class GPU. Final paper evidence still needs at least one full-precision DLM replication.

Embedded payload SHA-256: `6afff1a626f89cb5985591a7ea596796e368546b2425a6184bd7b09b97b896a8`

In [ ]:
!pip -q install -U "transformers>=4.57,<5" "accelerate>=1.0" "bitsandbytes>=0.47" pandas scipy

## 1. Runtime preflight

In [ ]:
import os, gc, json, math, re, time, random, hashlib, platform, ast as pyast
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

import numpy as np
import pandas as pd
import torch
import transformers
from scipy.stats import binomtest

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

BASE_SEED = 3400
random.seed(BASE_SEED)
np.random.seed(BASE_SEED)
torch.manual_seed(BASE_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(BASE_SEED)
    torch.cuda.empty_cache()

gc.collect()

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())

assert torch.cuda.is_available(), (
    "Notebook 12 requires a CUDA GPU runtime."
)

GPU_NAME = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30

print("GPU:", GPU_NAME)
print("VRAM:", round(GPU_GB, 2), "GB")
print("Allocated:", round(torch.cuda.memory_allocated()/2**30, 3), "GB")

if GPU_GB < 14:
    raise RuntimeError(
        "This exploratory 4-bit Dream self-repair experiment expects "
        "approximately a T4-class GPU (~15 GB) or larger."
    )

if torch.cuda.memory_allocated()/2**30 > 1.0:
    raise RuntimeError(
        "GPU memory is already occupied. Restart the runtime and run from the top."
    )

print("Runtime preflight: PASS")

## 2. Recover and validate the exact Dream-origin self-repair cohort

In [ ]:
import base64, zlib

EMBEDDED_PAYLOAD_SHA256 = "6afff1a626f89cb5985591a7ea596796e368546b2425a6184bd7b09b97b896a8"

_raw = zlib.decompress(
    base64.b64decode('''eNrlXAtv20a2/isDocDeTWSVw8dwaFzfRXbbYrvothdJeouLdaGOyJFFmCJVPqR4u/nv+50hKVKUbMuy3XiRNk4s8sx5zcx3Hhzqt1GhEx2WOprOVZxUuS5G5//4bXSlU52rMs7SaRyNzkdhVqVllG3S81Al8ay+dc6l5Z271r+iXKvlvxzH4qPxaJVns0QvjxgH4kiVqtBln3I6hxILYqTyMiZ6utuNrkUsVzTox0KzcqFZWi1nOi+YHYwZ/qg0YpLpDyoskxuWpSGoMoZRURXWA0qVX+mScXvC3uqyylOm2BoyIqbyuFwsdRmHYLCCKgVkGo4aPxvcZL988+33b76bvvn+3U9fvz1n/11L/59fJlCtZjw6B2drDOdqeIEcMx5lVbmqSOn3GSuyZE2KxAVr3DVmG5gBctK0egKzLtPL9IdUs1UGCyCBSKOslok/sxtIidMrM2yeJUm2wadzGmXbAWNnjAX45zWDRHbBwJJuvcvGZsBdroqLc+JwRuNfY/RFM3bXaVAR3lrrPJ7HYT2v57+NijKPw3IKXhH+NRdrR8ITIA0P3Bk4vGZQVGEIdUbnZV7pbvTgMpbedXdtrpICF7t1CF5ViLVBU/jbYWLjh2ln/Og8rZJkPOquTEFS6fZ6kVV5qKfNvGKn2QG0ph85sX4eGhMudHiNnRli4mPslGZv9sWNhpM1MkqRRO5IYpLEJXZyYoTV0jpxWGdFq8yU1lPfsEpPwyzPAQ7N1Y/jA7Lbaf495XK7J80eCKMLDxFBS+Hjzx8/PsXawcasVDKdZ/lG5TRvCakk3HGLp/VkrlUxLbNrneJmIMajDeimhQ6zNMIV25oAH60gcKXvWK6QEoCn1fV0navl9Go2OvcmfiACO5COsH2HB854lFdYr/FVqmi9El7OHBX50gqk5v7cdy3fdoLACt3Z3A7nrudLby64CkNb2DMp/IhzbyasmZDhXEpbj8gdYAZ9Rtk1Puk8z/J2GTeh4oGhgebxAXHF9k6LK7b39HGFA33tBoFt9ygIdp8tsrhPGFlOM4zg/A9FuRs7DttnIgrnAfsv/Lxi9h+BGDYhBqQhMrj9qDKPU5XcHlUMpyA4wGAQWtwjI4t7a2RxdyKL+58YWTjBrQFe292NLe6xsWXH2T3IDdwh5gZDYY9F+L64PWnufzjCczmxheS+H0jBXc+yDwG87wbC4RJk3AleKsDvYvQDAZ7z0wCe86cHeDFm3HpQhi2fr3KQdwH8YWV6yfy9JpmAgEW6DQbHQPjl5T+YYJeXZbzUBcDcoqTvj7jwM939Tpd/KMAzCasEe8DEn7uGXfQuclMiWLLlNYgGabHReQv/A/9h1C/HYX29rA9ivSV3ywh5CO2b7b8HFc31F4D3Ampz60AlIY+Fe4H43M5QD36tYTo/kHQAh2sAPQrpSSiX94p7mrrBulOQufJwKQcjyynr5amqB8sJbM+Tnu8KX3gHY4vNHcv3PdfjQrzU2LIbHh4YW8RpoUU8Q+kADOaixmHOj8uwg+erHYKHxhY3OBBb7jDq1OgCNl2YEAAhblGtwPnlJWtDA8jGrEdoi4aiu2vL/WuwofloXISPjw4atR+Hbj0SBF5OjUAozg3A8kGNEBxdJHATNvoz1oGs7XtDkOVbkVYn97HATjo0i+Fe4bZ4MrntYrtLJmhOlveg0PI79KV8j3vUefKFa0v7cNmCqsYTliUDzvlLDS3ivshyVSzlNTyq4vQc9Zd/RzQZku5EEHPzuLjxt2yRspm6pozYZmEGr1clW6pQ5VmWFmOmVbgAtsZXC8JSj8E8TPKE/dXAbMpWKrwugMhM/4rJbqCaZXMDwls+LE6Zy6J4Pte5Tks2y+EhyL2CBNgY3QCvcxbpJAZA3kzYT8QaDGZ5rOeA+ESrNVQkntdxGdJdimJxeG2urRZZihBA7SfsgTKhgRlu5OxdqdcaRpT1YMiOcwSKLC3xa0F6YmSrbqfVhEEFVdIjCxOVshK2GS+UY7Kl9sK4HZjrJSaCHLTnwT/14qDr3RoGT5yHMSuyPQ0PeN9wfYVhKIGs/hw+fvqggNENn4xf4Qgj7UtGnTKnYzI5dTJaI0Gr8NNjiXUxLyePDatOP6g2c/RshRip8aIydz4RABBhAWE91wEW7uMrgJPjvgSCetz/tKn7XRh5P6YGwdGYGgSnYuo3gDJNOMZm8bXGLkvxa6ExO2PCNOqRKyTXX0i2zHKCUZX2rxOwQYW6z00cJuzbecMFS7RgX3j7NOCcbdiywkaMMoAILlMe3TIk6SZtrkdJM2YHm7h7e4P/0xhUNjGgJXlNAi7YF9zZWrRlSFk2Kol5DBOhdueMdvzdjjDVw7JKyniFYAMy8t0WVk2tQGJfsVoBa+f5AdF+Qa0ic2fy+M7Q4DGA+5kBkjVxfJfjf88VgRDewV6CdFxhubbASiXEejGAtAsw9wGS4J48EpCI9PQkr9Ar7AWFSI9cZZ7HV6RSlqOUZhtMS459Yv5BnKckapGhZKjrbEoN6KgOFgHL1pR3UV6EuSoqqoVXGfVJcaVLDmieEezfb7MStaSFR3F9ifh+U3NodFqpm8LkfnVCoPMrOiGiIzopUmdWnbZ1NpZr8DV1ON1veFMO2OdUq9Vn1DfT4I8RuGMZlUnNyE2WlwuAjmWRSxQpXi6w3TdxCFa36V+PHUozfMc7beqBZ+7VvgDKqIiAbKetAvVuxey6Nx7pNGskNrJu1/s2bzEsmw9mPtPxw3w+nDznw+T9CV7fLGIgOU3+ye436kMdhKcxsz8QipPz2LssGTPz0fbMx/GDfPVYrCeh1oEJ/ZyyT+kDx7mUwnVdKzhQ3UsONwWeI3zX4cT/hYD9ELzvA3tYcGz2SaSngv3fVVk2UA9PUhW5ShTKMqKroRc7ibbYXCNxwU5tf0fNy1mBMjCn7mlmALvmUJjO6hLVKYjZFaam6N3EOMBpjITJlK/2PgVlVPXo4S02q0oDMdTRZddX+3dMmml2H7Kr/STXsJ3DdhO3Okv7qa3Dm312uAG9MGWlauvOnrcaR70yf18wcMF/WxcZl73LSIutexoSGmFOIJqPh1w2GLel/JL8hyLdfN7zxt64mu6xMFRz6SNR57LPCYoc33d8lMBUCQeeOFQIC0+gBhaO57vUiXwhUDSElkNQVKzVctXQhwuVnEl5V4PxEPkOJBmC4yDpK5WmN8gZEwr8BZtlpkUXqlXbR8z1hqmNumGi3YVZElEzqKDU0uxKlV9TIoBx5uFOBcDAHuuxYimY0JAJ+x4oASoCQbFL0+S3jSpQEwoQpCjoZ2rXPnGEkjEyqpu6sq+JKXHLBfIDEtQZsNPtu72grh2yUNG9+jE1JxQkBUzfz9qaaTATkpuHXI33GrfVqYzxgZFx1gDSrjTqnjUCWsswMKZnhIvhtDSMx537zTN7QVzFDteen7ZqNCPouYWo9djzdn/Y42vonZbe51dBB4EjXeEiXfIdOse1D2S2ZUsCM8uT3qc96HUfMB0FZoEbPATMAjc4Fcy+q66pGotTVADcFSgIYmpeY+eaNnhuVjpQTIFoWbe66qzihgZ4fk1R1CRYMuu4puyAqOFI8IORRUg7JU7rXhMI+xhj24F95ysfBrm6grPr7zdCjDVGRLRzYrdrhm2fv/dGLbYe2DV7NqRvbN06oGnUkSoNqwt2wbnbvefRU4gpc6fn4dYFkwPHb4n0yBO4RHrrIdz25r6LPxsA4cEE+CEd35aB5BY/+CqA59MTAc6l51jOywKQARgcBSCu3xtj3wsgID8VQNonpxtqlpmWdN2gyXKVXiGuzzQF9IXK17rAdJkUqY6f1Nk2FRzIb1oK5qBqM0/xaL/Fq45Rc7p+e7NKd26vsEUjddPLgG6j3MR1r51krnWTMbgejTbkjSY7jxMsuVty2Ufi0m067Ag9BqpqRls3tubuYxQxNIDS+upLIqOfV1sjzVFRq75NpAPtzKWOYl99Q0DwdpvvxrV/jaeNk+tzVZYkrr/cxvbRiZJs52lv3j6nbMm3fCl4YFmOHXgHsyVLcB4ADi2gonxZYDcAruOyJeE8BOxAfirYvTHrXeM6FSnUks1LlFpaNYAyBME/V+E1SBdqiUogUaFuOrdj9hZphk7oJEhOiYfBQeyjdVzEZZZTi2ppjjYe5MKyOnnA2l/HWVXUwIfUTNoBkCFboQoxnHR0CwM67NhkICsF1JXNFt6C51aRu/m0Tbhot1j0pXsvXG4fbJ4mbgc0i2qGCQ7LARZuuZqW2tBj3dUham/H3eKmXt63JW37bX1vXtCEXKb/N6TpK3GBaSd2HVFnY80AVaZLR13g1P3j+74fPPr0Pnj0QbOdvM+pV2ZxRzgeR/WITNE+WGF6gccd6bmBtLwXliDu4t/Hn8fNFxIQZC51qQjrzKHT0945fcKvI2hOz/YgvV/qIakp4oKOT7VfpNCeyGwVNxynKsyzAjO6yaYNQ+JCM0nx4rT3rp7w3diXZOSzvR/2kox8phcVfncTn/rE7Cc24PHH0z6pAU9xnOWTGvAUj2h/dwOe68HOpzfkiZq6n96QJ2ou3WGIfdiQJuk70hJ+xJQ8TeX4/JYgpaOjT7Msu+Z8eoW8lrL8TmCT11Zp/Gu1o0gcNbdM+tx9PZVnbw0pplTmdLo5Wz50CQl05xRaH3LPoj0SpN/LOI2X1XLaJOjpXNdp+a+VSsv4n02hMpqlM9xzp+4sLuklrCQzx6voKNJXb79+8/fpDz99P/3mzbff/fj26+lffnj7vz++m+L6V//ffCPBjltQScZzXZTkmjDXJvmvypDeMbJsccatM+68t2xMy7nLJ8KWMgheW9a5OXLwDAk9nFJmYZZM1+QvY69ZbGdIMs7aqThbExRolJ5RRI5UN0mmommxULYnMGLuh8p1lOVwGXpCuKHms8iRgeN4M+ignNCVMwvlCWqTecBtobgIPB1ZoVaOP6fvgFhmkU7qJf1VLT+/+rL+bW2dfZvWtdmZ/+ctLRWojcaW5zhuOAvmSs19gTBhR+E8iKTv+KFw7Zm2Z7YSiuw9fm4HO4Ve8TLwRRsGS53O+cX0viTKRr1qCrql+jBN9Wanjiv1ckWMzJRZE6zpMltNV/R7gCJNJVfQA8s0z1YkFp+nNITuo8K9WlXUj9BFoth7chQuTE0vwFR93J14wqGvjsD0cul7xD0PF7SeJvQu2uuw4g6tHUBEWqAGXZo3AkcY50/E6GO/DGsAwbWsj/8G8B5mvg==''')
)

assert hashlib.sha256(_raw).hexdigest() == EMBEDDED_PAYLOAD_SHA256

UPSTREAM = json.loads(_raw.decode("utf-8"))

FAILURES = UPSTREAM["selected_failures"]
SELECTION_META = UPSTREAM["selection_metadata"]
UPSTREAM_GATE = UPSTREAM["notebook11_gate"]

assert len(FAILURES) == 12
assert len({x["problem_id"] for x in FAILURES}) == 12
assert Counter(x["dataset"] for x in FAILURES) == {
    "countdown_fresh": 4,
    "gsm8k": 4,
    "svamp": 4,
}
assert all(not x["task_success"] for x in FAILURES)
assert UPSTREAM_GATE["DREAM_OWN_FAILURE_CORPUS_READY"] is True

print("Payload SHA-256:", EMBEDDED_PAYLOAD_SHA256)
print("Selected Dream failures:", len(FAILURES))
print("By dataset:", dict(Counter(x["dataset"] for x in FAILURES)))
print("Persistent selected problems:", sum(x["persistent_failure"] for x in SELECTION_META))

for row in SELECTION_META:
    print(
        row["failure_id"],
        "|",
        row["dataset"],
        "| persistent =",
        row["persistent_failure"],
    )

print("Dream-origin cohort validation: PASS")

## 3. Strict and numeric verifiers

In [ ]:
FINAL_RE = re.compile(
    r"(?im)^[ \t]*FINAL_ANSWER[ \t]*:[ \t]*"
    r"([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)"
    r"[ \t]*$"
)

NUM_RE = re.compile(
    r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?"
)


def close_num(a, b):
    return (
        a is not None
        and math.isclose(
            float(a),
            float(b),
            abs_tol=1e-8,
            rel_tol=1e-8,
        )
    )


def strict_final(text):
    vals = FINAL_RE.findall(str(text))

    if not vals:
        return None

    try:
        x = float(vals[-1])
        return x if math.isfinite(x) else None
    except Exception:
        return None


def last_number(text):
    vals = NUM_RE.findall(
        str(text).replace(",", "")
    )

    if not vals:
        return None

    try:
        x = float(vals[-1])
        return x if math.isfinite(x) else None
    except Exception:
        return None


def basic_verification(text, target):
    strict = strict_final(text)
    loose = last_number(text)
    target = float(target)

    return {
        "strict_prediction": strict,
        "numeric_prediction": loose,
        "target": target,
        "strict_success": bool(close_num(strict, target)),
        "numeric_success": bool(close_num(loose, target)),
    }


assert basic_verification(
    "reasoning\nFINAL_ANSWER: 4",
    4,
)["strict_success"]

assert basic_verification(
    "The result is 4.",
    4,
)["numeric_success"]

print("Strict/numeric verifier tests: PASS")

## 4. Countdown task-aware verifier

In [ ]:
COUNTDOWN_PROMPT_RE = re.compile(
    r"Use the numbers\s+(.+?)\s+exactly once\s+"
    r"to produce the target\s+"
    r"([-+]?(?:\d+(?:\.\d*)?|\.\d+))",
    flags=re.IGNORECASE,
)

ARITH_RUN_RE = re.compile(
    r"[0-9eE+\-*/().=\s]{3,}"
)

ALLOWED_BINOPS = (
    pyast.Add,
    pyast.Sub,
    pyast.Mult,
    pyast.Div,
)

ALLOWED_UNARY = (
    pyast.UAdd,
    pyast.USub,
)


def parse_countdown_prompt(prompt):
    m = COUNTDOWN_PROMPT_RE.search(str(prompt))

    if m is None:
        raise ValueError(
            "Countdown prompt format was not recognized."
        )

    source_nums = [
        float(x)
        for x in NUM_RE.findall(m.group(1))
    ]

    target = float(m.group(2))

    return source_nums, target


def safe_arith_tree_value_and_literals(expr):
    try:
        tree = pyast.parse(expr, mode="eval")
    except Exception:
        return None

    literals = []

    def walk(node):
        if isinstance(node, pyast.Expression):
            return walk(node.body)

        if isinstance(node, pyast.Constant):
            if not isinstance(node.value, (int, float)):
                raise ValueError

            value = float(node.value)

            if not math.isfinite(value):
                raise ValueError

            literals.append(value)
            return value

        if isinstance(node, pyast.BinOp) and isinstance(node.op, ALLOWED_BINOPS):
            a = walk(node.left)
            b = walk(node.right)

            if isinstance(node.op, pyast.Add):
                return a + b

            if isinstance(node.op, pyast.Sub):
                return a - b

            if isinstance(node.op, pyast.Mult):
                return a * b

            if isinstance(node.op, pyast.Div):
                if abs(b) < 1e-15:
                    raise ZeroDivisionError
                return a / b

        if isinstance(node, pyast.UnaryOp) and isinstance(node.op, ALLOWED_UNARY):
            value = walk(node.operand)

            if isinstance(node.op, pyast.USub):
                return -value

            return value

        raise ValueError

    try:
        value = walk(tree)

        if not math.isfinite(value):
            return None

        return {
            "value": float(value),
            "literals": literals,
        }

    except Exception:
        return None


def number_multiset_equal(a, b):
    if len(a) != len(b):
        return False

    aa = sorted(float(x) for x in a)
    bb = sorted(float(x) for x in b)

    return all(
        close_num(x, y)
        for x, y in zip(aa, bb)
    )


def countdown_candidates(text):
    text = (
        str(text)
        .replace("×", "*")
        .replace("÷", "/")
        .replace("\\times", "*")
        .replace("\\div", "/")
    )

    candidates = []

    for m in ARITH_RUN_RE.finditer(text):
        run = m.group(0).strip()

        if not any(op in run for op in ["+", "-", "*", "/"]):
            continue

        parts = [
            p.strip()
            for p in run.split("=")
            if p.strip()
        ]

        candidates.extend(parts)

    return sorted(
        set(candidates),
        key=len,
        reverse=True,
    )


def countdown_task_verification(text, prompt):
    source_nums, target = parse_countdown_prompt(prompt)

    checked = []

    for expr in countdown_candidates(text):
        parsed = safe_arith_tree_value_and_literals(expr)

        if parsed is None:
            continue

        uses_numbers_once = number_multiset_equal(
            parsed["literals"],
            source_nums,
        )

        value_correct = close_num(
            parsed["value"],
            target,
        )

        checked.append({
            "expression": expr,
            "value": parsed["value"],
            "literals": parsed["literals"],
            "uses_numbers_once": uses_numbers_once,
            "value_correct": value_correct,
        })

        if uses_numbers_once and value_correct:
            return {
                "task_success": True,
                "valid_expression": expr,
                "expression_value": parsed["value"],
                "source_numbers": source_nums,
                "target": target,
                "checked_candidates": checked,
            }

    return {
        "task_success": False,
        "valid_expression": None,
        "expression_value": None,
        "source_numbers": source_nums,
        "target": target,
        "checked_candidates": checked,
    }


_demo_prompt = (
    "Use the numbers 19, 29, and 24 exactly once "
    "to produce the target 14. "
    "Return a valid arithmetic expression."
)

assert countdown_task_verification(
    "The valid expression is 19 + 24 - 29 = 14.",
    _demo_prompt,
)["task_success"]

assert not countdown_task_verification(
    "29 - 19 = 10. FINAL_ANSWER: 14",
    _demo_prompt,
)["task_success"]

print("Countdown task verifier tests: PASS")

## 5. Unified self-repair verifier

In [ ]:
def verify_repair(text, failure):
    basic = basic_verification(
        text,
        failure["target"],
    )

    if failure["dataset"] == "countdown_fresh":
        structured = countdown_task_verification(
            text,
            failure["prompt"],
        )

        task_success = bool(
            structured["task_success"]
        )

    else:
        structured = None
        task_success = bool(
            basic["numeric_success"]
        )

    return {
        **basic,
        "task_success": task_success,
        "countdown_structured": structured,
    }


# Every selected starting response must remain a genuine failure
# under the exact verifier used for self-repair.
for failure in FAILURES:
    result = verify_repair(
        failure["output"],
        failure,
    )

    assert not result["task_success"], (
        "Selected starting Dream response unexpectedly passes: "
        + failure["failure_id"]
    )

print("All 12 starting Dream responses remain task failures: PASS")

## 6. Self-repair configuration

In [ ]:
MODEL_ID = "Dream-org/Dream-v0-Instruct-7B"
MODEL_REVISION = "05334cb9faaf763692dcf9d8737c642be2b2a6ae"

ACTIONS = [
    "open_last_25",
    "open_last_50",
    "open_last_75",
    "global",
]

REPAIR_SEEDS = [
    3401,
    3402,
    3403,
    3404,
    3405,
]

DENOISING_STEPS = 64
TEMPERATURE = 0.2
TOP_P = 0.95
ALG = "entropy"
ALG_TEMP = 0.0

EXPECTED_TRIALS = (
    len(FAILURES)
    * len(ACTIONS)
    * len(REPAIR_SEEDS)
)

assert EXPECTED_TRIALS == 240

print("Dream-origin failures:", len(FAILURES))
print("Actions:", ACTIONS)
print("Repair seeds:", REPAIR_SEEDS)
print("Expected self-repair trials:", EXPECTED_TRIALS)

## 7. Load Dream-7B in exploratory NF4 mode

In [ ]:
from transformers import (
    AutoModel,
    AutoTokenizer,
    BitsAndBytesConfig,
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

torch.cuda.empty_cache()
gc.collect()
torch.cuda.reset_peak_memory_stats()

t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=True,
)

model = AutoModel.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map={"": 0},
    low_cpu_mem_usage=True,
).eval()

assert hasattr(model, "diffusion_generate")

MODEL_DEVICE = next(model.parameters()).device

print("MODEL LOAD: PASS")
print("Class:", type(model).__name__)
print("Device:", MODEL_DEVICE)
print("Load seconds:", round(time.time() - t0, 2))
print(
    "Allocated GB:",
    round(torch.cuda.memory_allocated()/2**30, 3),
)

## 8. Prompt and failed-response canvas helpers

In [ ]:
SYSTEM_PROMPT = (
    "Solve the quantitative problem carefully. "
    "Give a mathematically correct answer. "
    "When possible, end with `FINAL_ANSWER: <number>`."
)


def encode_problem_prompt(problem):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": problem,
        },
    ]

    encoded = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    return {
        "input_ids": encoded.input_ids.to(MODEL_DEVICE),
        "attention_mask": encoded.attention_mask.to(MODEL_DEVICE),
    }


def encode_failed_response(text):
    ids = tokenizer(
        text,
        add_special_tokens=False,
        return_tensors="pt",
    ).input_ids[0]

    if len(ids) < 4:
        raise RuntimeError(
            "Dream failure retokenized to fewer than four tokens."
        )

    return ids


for failure in FAILURES:
    ids = encode_failed_response(
        failure["output"]
    )

    print(
        failure["failure_id"],
        "| retokenized response tokens =",
        len(ids),
    )

print("Dream failure canvas preparation: PASS")

## 9. Selective-remasking scopes

In [ ]:
def force_mask_for_action(length, action):
    mask = torch.zeros(
        length,
        dtype=torch.bool,
    )

    if action == "global":
        return mask

    preserve_fraction = {
        "open_last_25": 0.75,
        "open_last_50": 0.50,
        "open_last_75": 0.25,
    }[action]

    cut = max(
        1,
        int(math.floor(length * preserve_fraction)),
    )

    mask[:cut] = True

    return mask


for length in [8, 17, 40]:
    for action in ACTIONS:
        m = force_mask_for_action(length, action)

        print(
            "length",
            length,
            "|",
            action,
            "| forced",
            int(m.sum()),
            "| open",
            int((~m).sum()),
        )

print("Selective-remasking scope audit: PASS")

## 10. Actual forward-call instrumentation

In [ ]:
class ForwardCounter:
    def __init__(self, module):
        self.module = module
        self.calls = 0
        self.handle = None

    def _hook(self, module, inputs):
        self.calls += 1

    def __enter__(self):
        self.calls = 0
        self.handle = self.module.register_forward_pre_hook(
            self._hook
        )
        return self

    def __exit__(self, exc_type, exc, tb):
        if self.handle is not None:
            self.handle.remove()
            self.handle = None

## 11. Dream → Dream self-repair runner

In [ ]:
def reseed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


@torch.inference_mode()
def run_self_repair(
    failure,
    action,
    seed,
):
    reseed(seed)

    prompt = encode_problem_prompt(
        failure["prompt"]
    )

    original_ids = (
        encode_failed_response(
            failure["output"]
        )
        .to(MODEL_DEVICE)
    )

    response_len = int(
        original_ids.numel()
    )

    force_mask_cpu = force_mask_for_action(
        response_len,
        action,
    )

    force_mask = force_mask_cpu.to(
        MODEL_DEVICE
    )

    prompt_len = int(
        prompt["input_ids"].shape[1]
    )

    def clamp_hook(step, x, logits):
        generated = x[
            :,
            prompt_len:
            prompt_len + response_len
        ]

        if force_mask.any():
            generated[
                :,
                force_mask
            ] = original_ids[
                force_mask
            ].unsqueeze(0)

        return x

    torch.cuda.reset_peak_memory_stats()

    t0 = time.time()

    with ForwardCounter(model) as fc:
        output = model.diffusion_generate(
            prompt["input_ids"],
            attention_mask=
                prompt["attention_mask"],
            max_new_tokens=
                response_len,
            output_history=False,
            return_dict_in_generate=True,
            steps=
                DENOISING_STEPS,
            temperature=
                TEMPERATURE,
            top_p=
                TOP_P,
            alg=
                ALG,
            alg_temp=
                ALG_TEMP,
            generation_tokens_hook_func=
                clamp_hook,
        )

    wall = time.time() - t0

    final_ids = output.sequences[
        0,
        prompt_len:
        prompt_len + response_len
    ].detach()

    if force_mask.any():
        assert torch.equal(
            final_ids[force_mask],
            original_ids[force_mask],
        ), (
            "A mechanically preserved Dream token changed."
        )

    repaired = tokenizer.decode(
        final_ids.tolist(),
        skip_special_tokens=True,
    ).strip()

    verification = verify_repair(
        repaired,
        failure,
    )

    final_cpu = final_ids.cpu()
    original_cpu = original_ids.cpu()

    token_position_preservation = float(
        (
            final_cpu
            == original_cpu
        )
        .float()
        .mean()
        .item()
    )

    forced_fraction = float(
        force_mask_cpu
        .float()
        .mean()
        .item()
    )

    original_numeric = last_number(
        failure["output"]
    )

    repaired_numeric = verification[
        "numeric_prediction"
    ]

    same_wrong_numeric = bool(
        not verification["task_success"]
        and original_numeric is not None
        and repaired_numeric is not None
        and close_num(
            original_numeric,
            repaired_numeric,
        )
    )

    return {
        "failure_id":
            failure["failure_id"],
        "problem_id":
            failure["problem_id"],
        "dataset":
            failure["dataset"],
        "source_generation_seed":
            int(failure["seed"]),
        "action":
            action,
        "repair_seed":
            int(seed),
        "response_tokens":
            response_len,
        "forced_tokens":
            int(force_mask_cpu.sum()),
        "open_tokens":
            int((~force_mask_cpu).sum()),
        "forced_fraction":
            forced_fraction,
        "strict_success":
            bool(
                verification["strict_success"]
            ),
        "numeric_success":
            bool(
                verification["numeric_success"]
            ),
        "task_success":
            bool(
                verification["task_success"]
            ),
        "same_wrong_numeric":
            same_wrong_numeric,
        "verification":
            verification,
        "original_output":
            failure["output"],
        "repaired_output":
            repaired,
        "token_position_preservation":
            token_position_preservation,
        "actual_forward_calls":
            int(fc.calls),
        "wall_seconds":
            float(wall),
        "peak_vram_gb":
            float(
                torch.cuda.max_memory_allocated()
                / 2**30
            ),
    }

## 12. One real Dream self-repair smoke test

In [ ]:
smoke = run_self_repair(
    FAILURES[0],
    "open_last_25",
    REPAIR_SEEDS[0],
)

print(
    json.dumps(
        {
            k: v
            for k, v in smoke.items()
            if k not in {
                "original_output",
                "repaired_output",
                "verification",
            }
        },
        indent=2,
    )
)

print("\nORIGINAL DREAM FAILURE\n")
print(smoke["original_output"])

print("\nDREAM SELF-REPAIR\n")
print(smoke["repaired_output"])

assert smoke["actual_forward_calls"] >= DENOISING_STEPS
assert math.isfinite(smoke["wall_seconds"])

print("\nDREAM SELF-REPAIR SMOKE: PASS")

## 13. Output directory and immutable run signature

In [ ]:
SAVE_TO_DRIVE = True
CLEAR_OLD_RESULTS = False

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

    OUT_DIR = Path(
        "/content/drive/MyDrive/RepairScopeX/Notebook12_Dream_Self_Repair"
    )

else:
    OUT_DIR = Path(
        "/content/RepairScopeX_Notebook12_Dream_Self_Repair"
    )

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

signature_payload = {
    "protocol_version":
        "dream-self-repair-v1",
    "embedded_payload_sha256":
        EMBEDDED_PAYLOAD_SHA256,
    "model_id":
        MODEL_ID,
    "model_revision":
        MODEL_REVISION,
    "quantization":
        "bnb_nf4_4bit_exploratory",
    "actions":
        ACTIONS,
    "repair_seeds":
        REPAIR_SEEDS,
    "denoising_steps":
        DENOISING_STEPS,
    "temperature":
        TEMPERATURE,
    "top_p":
        TOP_P,
    "alg":
        ALG,
    "alg_temp":
        ALG_TEMP,
}

RUN_SIGNATURE = hashlib.sha256(
    json.dumps(
        signature_payload,
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()

TRIALS_PATH = (
    OUT_DIR
    / "dream_self_repair_trials.jsonl"
)

MANIFEST_PATH = (
    OUT_DIR
    / "run_manifest.json"
)

if CLEAR_OLD_RESULTS:
    for p in [
        TRIALS_PATH,
        MANIFEST_PATH,
    ]:
        if p.exists():
            p.unlink()

if MANIFEST_PATH.exists():
    old = json.loads(
        MANIFEST_PATH.read_text(
            encoding="utf-8"
        )
    )

    if old.get(
        "run_signature"
    ) != RUN_SIGNATURE:
        raise RuntimeError(
            "Existing Notebook 12 outputs use a different configuration."
        )

else:
    MANIFEST_PATH.write_text(
        json.dumps(
            {
                "created_utc":
                    datetime.now(
                        timezone.utc
                    ).isoformat(),
                "run_signature":
                    RUN_SIGNATURE,
                **signature_payload,
                "gpu":
                    GPU_NAME,
                "gpu_total_gb":
                    GPU_GB,
                "torch":
                    torch.__version__,
                "transformers":
                    transformers.__version__,
            },
            indent=2,
        ),
        encoding="utf-8",
    )

print("Run signature:", RUN_SIGNATURE)
print("Output:", OUT_DIR)

## 14. Resume-safe 240-trial self-repair sweep

In [ ]:
def trial_id(
    failure_id,
    action,
    repair_seed,
):
    return (
        failure_id
        + "|repair|"
        + action
        + "|"
        + str(repair_seed)
    )


EXPECTED_IDS = {
    trial_id(
        failure["failure_id"],
        action,
        repair_seed,
    )
    for failure in FAILURES
    for action in ACTIONS
    for repair_seed in REPAIR_SEEDS
}

assert len(EXPECTED_IDS) == 240


def load_completed(path):
    done = {}

    if not path.exists():
        return done

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as f:

        for line_number, line in enumerate(
            f,
            start=1,
        ):
            if not line.strip():
                continue

            try:
                rec = json.loads(line)
            except json.JSONDecodeError as e:
                raise RuntimeError(
                    f"Malformed JSONL at line {line_number}."
                ) from e

            if rec.get(
                "run_signature"
            ) != RUN_SIGNATURE:
                raise RuntimeError(
                    "Found trial from another Notebook 12 configuration."
                )

            tid = rec["trial_id"]

            if tid in done:
                raise RuntimeError(
                    f"Duplicate trial_id: {tid}"
                )

            done[tid] = rec

    return done


completed = load_completed(
    TRIALS_PATH
)

print(
    "Already completed:",
    len(completed),
    "/ 240",
)

counter = 0

with open(
    TRIALS_PATH,
    "a",
    encoding="utf-8",
) as fp:

    for failure in FAILURES:
        for action in ACTIONS:
            for repair_seed in REPAIR_SEEDS:
                counter += 1

                tid = trial_id(
                    failure["failure_id"],
                    action,
                    repair_seed,
                )

                if tid in completed:
                    print(
                        f"[{counter}/240] SKIP {tid}"
                    )
                    continue

                try:
                    rec = run_self_repair(
                        failure,
                        action,
                        repair_seed,
                    )

                    rec["trial_id"] = tid
                    rec["run_signature"] = RUN_SIGNATURE
                    rec["status"] = "ok"
                    rec["error"] = None

                except torch.cuda.OutOfMemoryError as e:
                    torch.cuda.empty_cache()
                    gc.collect()

                    rec = {
                        "trial_id":
                            tid,
                        "failure_id":
                            failure["failure_id"],
                        "problem_id":
                            failure["problem_id"],
                        "dataset":
                            failure["dataset"],
                        "action":
                            action,
                        "repair_seed":
                            repair_seed,
                        "status":
                            "oom",
                        "error":
                            str(e),
                        "run_signature":
                            RUN_SIGNATURE,
                    }

                except Exception as e:
                    rec = {
                        "trial_id":
                            tid,
                        "failure_id":
                            failure["failure_id"],
                        "problem_id":
                            failure["problem_id"],
                        "dataset":
                            failure["dataset"],
                        "action":
                            action,
                        "repair_seed":
                            repair_seed,
                        "status":
                            "error",
                        "error":
                            f"{type(e).__name__}: {e}",
                        "run_signature":
                            RUN_SIGNATURE,
                    }

                fp.write(
                    json.dumps(
                        rec,
                        ensure_ascii=False,
                    )
                )
                fp.write("\n")
                fp.flush()

                if rec["status"] == "ok":
                    print(
                        f"[{counter}/240] {tid} | "
                        f"task={rec['task_success']} | "
                        f"samewrong={rec['same_wrong_numeric']} | "
                        f"NFE={rec['actual_forward_calls']}"
                    )
                else:
                    print(
                        f"[{counter}/240] {tid} | "
                        f"{rec['status']} | {rec['error']}"
                    )

print("Dream self-repair sweep finished.")

## 15. Validate exactly 240 completed self-repair trials

In [ ]:
records = load_completed(
    TRIALS_PATH
)

missing = EXPECTED_IDS - set(records)
extra = set(records) - EXPECTED_IDS

print("Expected:", len(EXPECTED_IDS))
print("Found:", len(records))
print("Missing:", len(missing))
print("Extra:", len(extra))

if missing:
    raise RuntimeError(
        "Self-repair sweep incomplete. "
        "Rerun the sweep cell; completed trials will be skipped."
    )

if extra:
    raise RuntimeError(
        "Unexpected Notebook 12 trial IDs exist."
    )

assert len(records) == 240

rows = [
    records[tid]
    for tid in sorted(EXPECTED_IDS)
]

status_counts = Counter(
    x["status"]
    for x in rows
)

print("Statuses:", dict(status_counts))

if status_counts.get("oom", 0):
    raise RuntimeError(
        "At least one Dream self-repair trial OOMed."
    )

if status_counts.get("error", 0):
    raise RuntimeError(
        "At least one Dream self-repair trial errored."
    )

TRIAL_DF = pd.DataFrame(rows)

assert len(TRIAL_DF) == 240
assert TRIAL_DF["trial_id"].nunique() == 240
assert (
    TRIAL_DF["actual_forward_calls"]
    >= DENOISING_STEPS
).all()

forced_rows = TRIAL_DF[
    TRIAL_DF["forced_tokens"] > 0
]

assert (
    forced_rows["token_position_preservation"]
    + 1e-12
    >= forced_rows["forced_fraction"]
).all()

print("240-trial Dream self-repair matrix integrity: PASS")

## 16. Action-level self-repair summary

In [ ]:
ACTION_DF = (
    TRIAL_DF.groupby(
        "action",
        as_index=False,
    )
    .agg(
        trials=(
            "task_success",
            "size",
        ),
        strict_success_rate=(
            "strict_success",
            "mean",
        ),
        numeric_success_rate=(
            "numeric_success",
            "mean",
        ),
        task_success_rate=(
            "task_success",
            "mean",
        ),
        same_wrong_rate_all=(
            "same_wrong_numeric",
            "mean",
        ),
        mean_actual_nfe=(
            "actual_forward_calls",
            "mean",
        ),
        mean_preservation=(
            "token_position_preservation",
            "mean",
        ),
        mean_wall_seconds=(
            "wall_seconds",
            "mean",
        ),
        max_peak_vram_gb=(
            "peak_vram_gb",
            "max",
        ),
    )
)

failure_only_rows = []

for action, group in TRIAL_DF.groupby("action"):
    failed = group[
        ~group["task_success"]
    ]

    failure_only_rows.append({
        "action":
            action,
        "failed_trials":
            len(failed),
        "same_wrong_given_failure":
            float(
                failed["same_wrong_numeric"].mean()
            )
            if len(failed)
            else 0.0,
    })

FAIL_RECURRENCE_DF = pd.DataFrame(
    failure_only_rows
)

ACTION_DF = ACTION_DF.merge(
    FAIL_RECURRENCE_DF,
    on="action",
    how="left",
)

display(
    ACTION_DF.sort_values(
        [
            "task_success_rate",
            "mean_preservation",
        ],
        ascending=[
            False,
            False,
        ],
    )
)

## 17. Dataset × action self-repair summary

In [ ]:
DATASET_DF = (
    TRIAL_DF.groupby(
        [
            "dataset",
            "action",
        ],
        as_index=False,
    )
    .agg(
        trials=(
            "task_success",
            "size",
        ),
        strict_success_rate=(
            "strict_success",
            "mean",
        ),
        numeric_success_rate=(
            "numeric_success",
            "mean",
        ),
        task_success_rate=(
            "task_success",
            "mean",
        ),
        same_wrong_rate=(
            "same_wrong_numeric",
            "mean",
        ),
        mean_nfe=(
            "actual_forward_calls",
            "mean",
        ),
        mean_preservation=(
            "token_position_preservation",
            "mean",
        ),
    )
)

display(DATASET_DF)

## 18. Per-failure self-repair surface and Oracle Repair Headroom

In [ ]:
surface_rows = []

for (
    failure_id,
    problem_id,
    dataset,
    action,
), group in TRIAL_DF.groupby(
    [
        "failure_id",
        "problem_id",
        "dataset",
        "action",
    ]
):
    assert len(group) == 5

    surface_rows.append({
        "failure_id":
            failure_id,
        "problem_id":
            problem_id,
        "dataset":
            dataset,
        "action":
            action,
        "task_success_rate":
            float(
                group["task_success"].mean()
            ),
        "strict_success_rate":
            float(
                group["strict_success"].mean()
            ),
        "numeric_success_rate":
            float(
                group["numeric_success"].mean()
            ),
        "same_wrong_rate":
            float(
                group["same_wrong_numeric"].mean()
            ),
        "mean_nfe":
            float(
                group["actual_forward_calls"].mean()
            ),
        "mean_preservation":
            float(
                group["token_position_preservation"].mean()
            ),
    })

SURFACE_DF = pd.DataFrame(
    surface_rows
)

assert len(SURFACE_DF) == 48

fixed = (
    SURFACE_DF.groupby(
        "action",
        as_index=False,
    )
    .agg(
        mean_task_success=(
            "task_success_rate",
            "mean",
        ),
        mean_nfe=(
            "mean_nfe",
            "mean",
        ),
        mean_preservation=(
            "mean_preservation",
            "mean",
        ),
    )
    .sort_values(
        [
            "mean_task_success",
            "mean_nfe",
            "mean_preservation",
        ],
        ascending=[
            False,
            True,
            False,
        ],
    )
)

best_fixed = fixed.iloc[0]

oracle_rows = []

for failure_id, group in SURFACE_DF.groupby(
    "failure_id"
):
    best = group.sort_values(
        [
            "task_success_rate",
            "mean_nfe",
            "mean_preservation",
        ],
        ascending=[
            False,
            True,
            False,
        ],
    ).iloc[0]

    oracle_rows.append({
        "failure_id":
            failure_id,
        "problem_id":
            best["problem_id"],
        "dataset":
            best["dataset"],
        "oracle_action":
            best["action"],
        "oracle_task_success":
            float(
                best["task_success_rate"]
            ),
        "oracle_mean_nfe":
            float(
                best["mean_nfe"]
            ),
        "oracle_preservation":
            float(
                best["mean_preservation"]
            ),
    })

ORACLE_DF = pd.DataFrame(
    oracle_rows
)

best_fixed_mean = float(
    best_fixed["mean_task_success"]
)

oracle_mean = float(
    ORACLE_DF[
        "oracle_task_success"
    ].mean()
)

DLM_ORH = (
    oracle_mean
    - best_fixed_mean
)

oracle_action_distribution = (
    ORACLE_DF[
        "oracle_action"
    ]
    .value_counts(
        normalize=True
    )
    .to_dict()
)

print(
    "Best fixed action:",
    best_fixed["action"],
)

print(
    "Best fixed task success:",
    round(best_fixed_mean, 4),
)

print(
    "Oracle task success:",
    round(oracle_mean, 4),
)

print(
    "Dream self-repair ORH:",
    round(DLM_ORH, 4),
)

print(
    "Oracle action distribution:",
    oracle_action_distribution,
)

display(ORACLE_DF)

## 19. Cluster-bootstrap confidence intervals

Failures are the sampling clusters. Each bootstrap draw resamples the **12 Dream-origin failures with replacement** and keeps all five repair seeds within each selected failure/action condition.

This avoids pretending the 60 action-level generations are 60 independent benchmark items.

In [ ]:
def cluster_bootstrap_action(
    action,
    n_boot=20000,
    seed=12001,
):
    sub = TRIAL_DF[
        TRIAL_DF["action"] == action
    ].copy()

    failure_ids = sorted(
        sub["failure_id"].unique()
    )

    assert len(failure_ids) == 12

    by_failure = {
        fid: sub[
            sub["failure_id"] == fid
        ]["task_success"].astype(float).to_numpy()
        for fid in failure_ids
    }

    observed = float(
        sub["task_success"].mean()
    )

    rng = np.random.default_rng(seed)

    boots = np.empty(
        n_boot,
        dtype=float,
    )

    for i in range(n_boot):
        sampled = rng.choice(
            failure_ids,
            size=len(failure_ids),
            replace=True,
        )

        vals = np.concatenate(
            [
                by_failure[fid]
                for fid in sampled
            ]
        )

        boots[i] = float(
            vals.mean()
        )

    lo, hi = np.quantile(
        boots,
        [0.025, 0.975],
    )

    return (
        observed,
        float(lo),
        float(hi),
    )


ci_rows = []

for idx, action in enumerate(ACTIONS):
    est, lo, hi = cluster_bootstrap_action(
        action,
        n_boot=20000,
        seed=12001 + idx,
    )

    ci_rows.append({
        "action":
            action,
        "task_success":
            est,
        "ci95_low":
            lo,
        "ci95_high":
            hi,
    })

ACTION_CI_DF = pd.DataFrame(
    ci_rows
)

display(ACTION_CI_DF)

## 20. Selective-remasking vs global matched-NFE contrasts

All four actions use the same nominal 64 denoising steps.

For each selective action:

- compare paired task outcomes to `global` using the same failure and repair seed;
- calculate task-success difference;
- run an exact one-sided discordant-pair test;
- apply Holm correction across the three selective scopes.

Positive difference means selective repair outperformed global regeneration.

In [ ]:
contrast_rows = []

for action in [
    "open_last_25",
    "open_last_50",
    "open_last_75",
]:
    left = (
        TRIAL_DF[
            TRIAL_DF["action"] == action
        ][
            [
                "failure_id",
                "repair_seed",
                "task_success",
            ]
        ]
        .rename(
            columns={
                "task_success":
                    "selective_success"
            }
        )
    )

    right = (
        TRIAL_DF[
            TRIAL_DF["action"] == "global"
        ][
            [
                "failure_id",
                "repair_seed",
                "task_success",
            ]
        ]
        .rename(
            columns={
                "task_success":
                    "global_success"
            }
        )
    )

    paired = left.merge(
        right,
        on=[
            "failure_id",
            "repair_seed",
        ],
        how="inner",
    )

    assert len(paired) == 60

    s = paired[
        "selective_success"
    ].astype(int).to_numpy()

    g = paired[
        "global_success"
    ].astype(int).to_numpy()

    selective_only = int(
        np.sum(
            (s == 1)
            & (g == 0)
        )
    )

    global_only = int(
        np.sum(
            (s == 0)
            & (g == 1)
        )
    )

    discordant = (
        selective_only
        + global_only
    )

    if discordant == 0:
        p_value = 1.0
    else:
        p_value = float(
            binomtest(
                selective_only,
                discordant,
                p=0.5,
                alternative="greater",
            ).pvalue
        )

    contrast_rows.append({
        "selective_action":
            action,
        "selective_task_success":
            float(s.mean()),
        "global_task_success":
            float(g.mean()),
        "difference_vs_global":
            float(
                s.mean()
                - g.mean()
            ),
        "selective_only_discordant":
            selective_only,
        "global_only_discordant":
            global_only,
        "raw_one_sided_p":
            p_value,
    })

CONTRAST_DF = pd.DataFrame(
    contrast_rows
)

# Holm step-down adjustment.
order = np.argsort(
    CONTRAST_DF[
        "raw_one_sided_p"
    ].to_numpy()
)

m = len(CONTRAST_DF)
adjusted = np.empty(
    m,
    dtype=float,
)

running = 0.0

for rank, idx in enumerate(order):
    raw_p = float(
        CONTRAST_DF.iloc[
            idx
        ]["raw_one_sided_p"]
    )

    adj = min(
        1.0,
        (m - rank) * raw_p,
    )

    running = max(
        running,
        adj,
    )

    adjusted[idx] = running

CONTRAST_DF[
    "holm_p"
] = adjusted

display(CONTRAST_DF)

## 21. Self-repair hysteresis contrast

This reports the coarse numeric recurrence phenomenon separately from correctness.

For each action:

`same-wrong recurrence = P(repaired final numeric state = original failed numeric state | repair still fails)`

For Countdown this is a **coarse numeric-state measure**, not a complete structural-error identity test.

In [ ]:
HYSTERESIS_DF = (
    ACTION_DF[
        [
            "action",
            "task_success_rate",
            "mean_preservation",
            "failed_trials",
            "same_wrong_given_failure",
        ]
    ]
    .copy()
)

display(
    HYSTERESIS_DF.sort_values(
        "same_wrong_given_failure",
        ascending=False,
    )
)

## 22. Dream self-repair progression gate

The self-repair experiment is considered ready for confirmation when:

1. all 240 trials complete without runtime failure;
2. mechanical preservation is valid;
3. at least 20 repairs are task-successful overall;
4. self-repair succeeds on all 3 datasets;
5. at least 2 selective scopes (`open_last_*`) have non-zero task success;
6. at least 10 task successes come from selective-remasking conditions rather than global regeneration.

The gate does **not** require selective remasking to beat global regeneration. A null or negative selective-vs-global contrast is scientifically meaningful and should be reported rather than tuned away.

In [ ]:
successful_repairs = int(
    TRIAL_DF[
        "task_success"
    ].sum()
)

datasets_with_success = int(
    TRIAL_DF.loc[
        TRIAL_DF[
            "task_success"
        ],
        "dataset",
    ].nunique()
)

selective = TRIAL_DF[
    TRIAL_DF[
        "action"
    ].str.startswith(
        "open_last_"
    )
]

selective_successes = int(
    selective[
        "task_success"
    ].sum()
)

selective_actions_with_success = int(
    selective.loc[
        selective[
            "task_success"
        ],
        "action",
    ].nunique()
)

preservation_ok = bool(
    (
        forced_rows[
            "token_position_preservation"
        ]
        + 1e-12
        >=
        forced_rows[
            "forced_fraction"
        ]
    ).all()
)

SELF_REPAIR_GATE = {
    "trials":
        int(
            len(TRIAL_DF)
        ),
    "unique_trial_ids":
        int(
            TRIAL_DF[
                "trial_id"
            ].nunique()
        ),
    "task_successful_repairs":
        successful_repairs,
    "selective_task_successful_repairs":
        selective_successes,
    "datasets_with_task_success":
        datasets_with_success,
    "selective_actions_with_success":
        selective_actions_with_success,
    "best_fixed_action":
        str(
            best_fixed["action"]
        ),
    "best_fixed_task_success":
        best_fixed_mean,
    "oracle_task_success":
        oracle_mean,
    "dlm_oracle_repair_headroom":
        DLM_ORH,
    "mechanical_preservation_ok":
        preservation_ok,
    "minimum_actual_nfe":
        int(
            TRIAL_DF[
                "actual_forward_calls"
            ].min()
        ),
    "quantization":
        "bnb_nf4_4bit_exploratory",
}

SELF_REPAIR_GATE[
    "DREAM_SELF_REPAIR_SURFACE_READY"
] = bool(
    SELF_REPAIR_GATE[
        "trials"
    ] == 240
    and SELF_REPAIR_GATE[
        "unique_trial_ids"
    ] == 240
    and SELF_REPAIR_GATE[
        "mechanical_preservation_ok"
    ]
    and SELF_REPAIR_GATE[
        "minimum_actual_nfe"
    ] >= DENOISING_STEPS
    and SELF_REPAIR_GATE[
        "task_successful_repairs"
    ] >= 20
    and SELF_REPAIR_GATE[
        "datasets_with_task_success"
    ] == 3
    and SELF_REPAIR_GATE[
        "selective_actions_with_success"
    ] >= 2
    and SELF_REPAIR_GATE[
        "selective_task_successful_repairs"
    ] >= 10
)

print(
    json.dumps(
        SELF_REPAIR_GATE,
        indent=2,
    )
)

## 23. Save all Notebook 12 artifacts

In [ ]:
selection_path = (
    OUT_DIR
    / "self_repair_cohort.csv"
)

action_path = (
    OUT_DIR
    / "self_repair_action_summary.csv"
)

dataset_path = (
    OUT_DIR
    / "self_repair_dataset_summary.csv"
)

surface_path = (
    OUT_DIR
    / "self_repair_surface.csv"
)

oracle_path = (
    OUT_DIR
    / "self_repair_oracle.csv"
)

ci_path = (
    OUT_DIR
    / "self_repair_action_ci.csv"
)

contrast_path = (
    OUT_DIR
    / "selective_vs_global_contrasts.csv"
)

hyst_path = (
    OUT_DIR
    / "self_repair_hysteresis.csv"
)

gate_path = (
    OUT_DIR
    / "DREAM_SELF_REPAIR_GATE.json"
)

pd.DataFrame(
    SELECTION_META
).to_csv(
    selection_path,
    index=False,
)

ACTION_DF.to_csv(
    action_path,
    index=False,
)

DATASET_DF.to_csv(
    dataset_path,
    index=False,
)

SURFACE_DF.to_csv(
    surface_path,
    index=False,
)

ORACLE_DF.to_csv(
    oracle_path,
    index=False,
)

ACTION_CI_DF.to_csv(
    ci_path,
    index=False,
)

CONTRAST_DF.to_csv(
    contrast_path,
    index=False,
)

HYSTERESIS_DF.to_csv(
    hyst_path,
    index=False,
)

gate_path.write_text(
    json.dumps(
        SELF_REPAIR_GATE,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "Saved Notebook 12 artifacts:",
    OUT_DIR,
)

## 24. Artifact manifest

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


artifact_paths = [
    MANIFEST_PATH,
    TRIALS_PATH,
    selection_path,
    action_path,
    dataset_path,
    surface_path,
    oracle_path,
    ci_path,
    contrast_path,
    hyst_path,
    gate_path,
]

artifact_manifest = {
    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
    "run_signature":
        RUN_SIGNATURE,
    "embedded_payload_sha256":
        EMBEDDED_PAYLOAD_SHA256,
    "protocol_version":
        "dream-self-repair-v1",
    "files": {
        p.name:
            sha256_file(p)
        for p in artifact_paths
        if p.exists()
    },
}

artifact_manifest_path = (
    OUT_DIR
    / "artifact_manifest.json"
)

artifact_manifest_path.write_text(
    json.dumps(
        artifact_manifest,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        artifact_manifest,
        indent=2,
    )
)

## 25. ZIP and download all Notebook 12 results

In [ ]:
import shutil

DOWNLOAD_RESULTS = True

ZIP_BASE = Path(
    "/content/RepairScopeX_Notebook12_Dream_Self_Repair_results"
)

zip_path = shutil.make_archive(
    str(ZIP_BASE),
    "zip",
    root_dir=OUT_DIR,
)

print("Created:", zip_path)

print(
    "Size MB:",
    round(
        Path(
            zip_path
        ).stat().st_size
        / 1024**2,
        3,
    ),
)

if DOWNLOAD_RESULTS:
    from google.colab import files

    files.download(
        zip_path
    )

# Progression rule

If:

`DREAM_SELF_REPAIR_SURFACE_READY = true`

then RepairScope-X has direct evidence that a real DLM can repair **its own** failed generations under selective remasking.

At that point, do not simply add more Dream pilot notebooks. The next priority should be:

1. independently confirm the strongest self-repair findings with new failures/seeds;
2. reproduce the central phenomenon on a second genuine DLM;
3. obtain at least one full-precision DLM replication on a larger-memory GPU;
4. then freeze the core paper figures/tables.

Send back:

`RepairScopeX_Notebook12_Dream_Self_Repair_results.zip`